#**ASSIGNMENT 2**

# 001 — Numerically Stable Softmax (PyTorch)

**Difficulty:** Easy · **Topic:** PyTorch basics, broadcasting, numerical stability

## Problem

Implement a numerically stable softmax along an arbitrary dimension of a tensor.

Given a float tensor `x` and an integer `dim`, compute

```
softmax(x)_i = exp(x_i - max_along_dim(x)) / sum_along_dim(exp(x - max))
```

You **must not** call `torch.softmax`, `F.softmax`, `torch.nn.functional.softmax`, or `torch.log_softmax`. The goal is to practice broadcasting, `keepdim`, and basic reductions.

## Input

- `x: torch.Tensor` of floating dtype, shape `(*)`, any number of dims ≥ 1. May be on CUDA.
- `dim: int`, a valid dimension of `x` (possibly negative).

## Output

A tensor of the same shape and dtype as `x` whose values along `dim` sum to `1` and are non-negative.

## Constraints / Notes

- Must be numerically stable for inputs with very large magnitudes (e.g. `1e4`). A naïve `exp(x) / exp(x).sum()` will fail those tests.
- Preserve input dtype (`float16`, `float32`, `float64`, `bfloat16` may appear).
- Preserve device (CPU or CUDA).

## Examples

```
>>> x = torch.tensor([[1.0, 2.0, 3.0]])
>>> Solution().solve(x, dim=-1)
tensor([[0.0900, 0.2447, 0.6652]])
```

## What you are learning

- `tensor.max(dim=..., keepdim=True)`
- Broadcasting and why `keepdim` matters
- Why max-subtraction gives numerical stability

In [129]:
import torch


class Solution001:
    """Implement numerically stable softmax along `dim`.

    Do NOT use torch.softmax / F.softmax / torch.log_softmax.
    """

    def solve(self, x: torch.Tensor, dim: int) -> torch.Tensor:
        max_val = x.max(dim = dim,keepdim = True)[0]

        red_val = (x - max_val).to(x.device,dtype = x.dtype)

        exp = torch.exp(red_val).to(x.device,dtype = x.dtype)

        sigma = exp.sum(dim = dim,keepdim = True).to(x.device,dtype = x.dtype)

        final = (exp/sigma).to(x.device,dtype = x.dtype)

        return final


Test your code here (run both blocks below):

In [130]:
# @title
import torch

try:
    from server.ast_checks import find_forbidden
except ModuleNotFoundError:
    find_forbidden = lambda src, banned: []  # local fallback


_BANNED_CALLS = [
    "torch.softmax",
    "torch.log_softmax",
    "torch.nn.functional.softmax",
    "torch.nn.functional.log_softmax",
    "F.softmax",
    "F.log_softmax",
    "functional.softmax",
    "functional.log_softmax",
    "Tensor.softmax",
    "x.softmax",
]


def static_checks(user_source: str):
    issues = []
    hits = find_forbidden(user_source, _BANNED_CALLS)
    if hits:
        issues.append((False, f"Calls to {', '.join(hits)} are not allowed — implement softmax manually."))
    return issues


def _gen(g_seed: int):
    return torch.Generator().manual_seed(g_seed)


# ---- generator helpers ----------------------------------------------------

def _randn(shape, seed, dtype=torch.float32, device="cpu", scale=1.0):
    g = _gen(seed)
    t = torch.randn(*shape, generator=g, dtype=torch.float32) * scale
    return t.to(device=device, dtype=dtype)


def _case(name, *, inputs, tol=(1e-3, 1e-4), check=None):
    return {"name": name, "inputs": inputs, "tolerance": tol, "check": check}


# ---- invariant checks ------------------------------------------------------

def _make_sum_check(dim):
    def chk(out):
        s = out.to(torch.float32).sum(dim=dim)
        ok = torch.allclose(s, torch.ones_like(s), atol=5e-3)
        return (ok, None if ok else f"sums along dim={dim} are not 1 (min={s.min().item():.4g}, max={s.max().item():.4g})")
    return chk


# ---- test cases -----------------------------------------------------------


# ── local runner ────────────────────────────────────────────────────────────
def _run_tests_001():
    _tc = [
        _case("1D basic",
            inputs=lambda dev: {"x": _randn((7,), 1, device=dev), "dim": -1},
            check=_make_sum_check(-1),
        ),
        _case("2D last dim",
            inputs=lambda dev: {"x": _randn((4, 11), 2, device=dev), "dim": -1},
            check=_make_sum_check(-1),
        ),
        _case("2D first dim",
            inputs=lambda dev: {"x": _randn((5, 9), 3, device=dev), "dim": 0},
            check=_make_sum_check(0),
        ),
        _case("3D middle dim (negative)",
            inputs=lambda dev: {"x": _randn((2, 5, 7), 4, device=dev), "dim": -2},
            check=_make_sum_check(-2),
        ),

        # ---- numerical stability: dominates naive `exp(x)/exp(x).sum()` -----
        _case("large positive magnitudes (1e4)",
            inputs=lambda dev: {"x": _randn((3, 16), 5, device=dev, scale=1e4), "dim": -1},
            check=_make_sum_check(-1),
        ),
        _case("large negative magnitudes (-1e4)",
            inputs=lambda dev: {"x": -_randn((3, 16), 6, device=dev).abs() * 1e4, "dim": -1},
            check=_make_sum_check(-1),
        ),
        _case("mixed +/- extreme magnitudes",
            inputs=lambda dev: {
                "x": torch.cat([
                    torch.full((1, 8), 1e3, device=dev),
                    torch.full((1, 8), -1e3, device=dev),
                    _randn((1, 8), 7, device=dev, scale=500).to(dev),
                ], dim=0),
                "dim": -1,
            },
            check=_make_sum_check(-1),
        ),

        # ---- degenerate inputs ------
        _case("all-equal input (uniform output)",
            inputs=lambda dev: {"x": torch.full((3, 5), 42.0, device=dev), "dim": -1},
        ),
        _case("all-zero input",
            inputs=lambda dev: {"x": torch.zeros(4, 6, device=dev), "dim": -1},
        ),
        _case("single element along dim (should be all 1)",
            inputs=lambda dev: {"x": _randn((4, 1), 8, device=dev), "dim": -1},
            check=_make_sum_check(-1),
        ),
        _case("size-1 non-softmax dim",
            inputs=lambda dev: {"x": _randn((1, 4, 5), 9, device=dev), "dim": -1},
            check=_make_sum_check(-1),
        ),

        # ---- dtype preservation ----
        _case("float64 precision",
            inputs=lambda dev: {"x": _randn((6, 9), 10, dtype=torch.float64, device=dev), "dim": -1},
            tol=(1e-7, 1e-9),
        ),
        _case("float16 (moderate range)",
            inputs=lambda dev: {"x": _randn((4, 16), 11, device=dev, scale=3.0).to(torch.float16), "dim": -1},
            tol=(5e-3, 1e-3),
        ),
        _case("bfloat16 (wider exponent)",
            inputs=lambda dev: {"x": _randn((4, 16), 12, device=dev, scale=10.0).to(torch.bfloat16), "dim": -1},
            tol=(3e-2, 1e-2),
        ),

        # ---- non-contiguous / strided inputs ----
        _case("non-contiguous (transpose)",
            inputs=lambda dev: {"x": _randn((8, 5), 13, device=dev).transpose(0, 1), "dim": -1},
            check=_make_sum_check(-1),
        ),
        _case("strided slice",
            inputs=lambda dev: {"x": _randn((4, 40), 14, device=dev)[:, ::3], "dim": -1},
            check=_make_sum_check(-1),
        ),
        _case("permuted 4D tensor",
            inputs=lambda dev: {
                "x": _randn((2, 3, 4, 5), 15, device=dev).permute(0, 2, 3, 1),
                "dim": 2,
            },
            check=_make_sum_check(2),
        ),

        # ---- high-rank & non-standard dims ----
        _case("4D softmax along dim=2",
            inputs=lambda dev: {"x": _randn((2, 3, 4, 5), 16, device=dev), "dim": 2},
            check=_make_sum_check(2),
        ),
        _case("5D softmax along negative middle dim",
            inputs=lambda dev: {"x": _randn((2, 2, 3, 4, 5), 17, device=dev), "dim": -3},
            check=_make_sum_check(-3),
        ),

        # ---- check that outputs match dtype and device ----
        _case("dtype preserved (fp64 input, fp64 output)",
            inputs=lambda dev: {"x": _randn((3, 7), 18, dtype=torch.float64, device=dev), "dim": -1},
            tol=(1e-7, 1e-9),
        ),

        # ---- immutability: user must not mutate input ----
        # (handled by the runner cloning inputs before each call)
    ]
    sol = Solution001()
    passed = failed = 0
    for tc in _tc:
        name = tc["name"]
        inputs = tc["inputs"]("cpu")
        try:
            out = sol.solve(**inputs)
            check = tc.get("check")
            if check:
                ok, msg = check(out)
                if not ok:
                    print(f"  FAIL  {name}: {msg}"); failed += 1; continue
            ref = torch.softmax(inputs["x"].float(), dim=inputs["dim"]).to(inputs["x"].dtype)
            tol = tc.get("tolerance", (1e-3, 1e-4))
            if not torch.allclose(out.float(), ref.float(), atol=tol[0], rtol=tol[1]):
                print(f"  FAIL  {name}: values don't match reference"); failed += 1; continue
            print(f"  PASS  {name}"); passed += 1
        except NotImplementedError:
            print(f"  SKIP  {name}: not implemented yet"); failed += 1
        except Exception as e:
            print(f"  ERROR {name}: {e}"); failed += 1
    print(f"\n{passed}/{passed+failed} tests passed")


In [131]:
_run_tests_001()

  PASS  1D basic
  PASS  2D last dim
  PASS  2D first dim
  PASS  3D middle dim (negative)
  PASS  large positive magnitudes (1e4)
  PASS  large negative magnitudes (-1e4)
  PASS  mixed +/- extreme magnitudes
  PASS  all-equal input (uniform output)
  PASS  all-zero input
  PASS  single element along dim (should be all 1)
  PASS  size-1 non-softmax dim
  PASS  float64 precision
  PASS  float16 (moderate range)
  PASS  bfloat16 (wider exponent)
  PASS  non-contiguous (transpose)
  PASS  strided slice
  PASS  permuted 4D tensor
  PASS  4D softmax along dim=2
  PASS  5D softmax along negative middle dim
  PASS  dtype preserved (fp64 input, fp64 output)

20/20 tests passed


# 002 — Pairwise Euclidean Distance

**Difficulty:** Easy · **Topic:** NumPy broadcasting

## Problem

Given two 2-D arrays `X` of shape `(N, D)` and `Y` of shape `(M, D)`, compute the pairwise Euclidean distance matrix of shape `(N, M)`, where `output[i, j]` is the L2 distance between row `i` of `X` and row `j` of `Y`.

Use the identity

```
||x - y||^2 = ||x||^2 - 2 * x · y + ||y||^2
```

to compute all distances in a fully-vectorised, loop-free way:

```python
out = np.sqrt(
    (X**2).sum(axis=1, keepdims=True)   # (N, 1)
    - 2 * X @ Y.T                        # (N, M)
    + (Y**2).sum(axis=1)                 # (M,) broadcasts to (N, M)
)
```

You **must not** use `scipy.spatial.distance`, `sklearn`, or any explicit Python loop.

## Input

- `X: np.ndarray` — shape `(N, D)`, dtype `float64`
- `Y: np.ndarray` — shape `(M, D)`, dtype `float64`

## Output

`np.ndarray` of shape `(N, M)` and dtype `float64`, where `output[i, j] = ||X[i] - Y[j]||_2`.

## Constraints / Notes

- All distances are non-negative.
- When `X` and `Y` are the same array, the diagonal is zero.
- When `X` and `Y` are the same array, the matrix is symmetric.
- Numerical precision: due to floating-point cancellation, clamp the argument to `sqrt` at `0` before taking the square root (use `np.maximum(sq, 0)` or `np.sqrt(np.maximum(sq, 0.0))`).
- `N`, `M`, `D` can each be 1.

## Examples

```python
>>> X = np.array([[0.0, 0.0], [3.0, 4.0]])
>>> Y = np.array([[0.0, 0.0]])
>>> Solution().solve(X, Y)
array([[0.],
       [5.]])
```

## What you are learning

- Broadcasting `(N, 1) - (N, M) + (M,)` without explicit tiling
- Using `keepdims=True` vs. letting broadcasting handle the reduction axis
- The expand-square trick to avoid materialising `(N, M, D)` intermediate tensors

In [132]:
import numpy as np


class Solution002:
    """Compute the pairwise Euclidean distance matrix between rows of X and Y.

    Do NOT use scipy, sklearn, or any explicit Python loop.
    """

    def solve(self, X: np.ndarray, Y: np.ndarray) -> np.ndarray:
        # X: (N, D), Y: (M, D)  →  out: (N, M)  float64
        # Hint: use (X**2).sum(...) - 2*X@Y.T + (Y**2).sum(...)
        out = np.sqrt( (X**2).sum(axis=1, keepdims=True)   - 2 * X @ Y.T  + (Y**2).sum(axis=1))

        if X is Y or np.array_equal(X, Y):

          np.fill_diagonal(out, 0.0)

        return out


Test your code here (run both blocks below):

In [133]:
# @title
import numpy as np


# No static_checks for this problem (loop detection unreliable in AST).


def _rng(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)


def _case(name, *, inputs, tol=(1e-9, 1e-9), check=None):
    return {"name": name, "inputs": inputs, "tolerance": tol, "check": check}


# ---- invariant checks -------------------------------------------------------

def _check_nonneg(out):
    if np.any(out < -1e-9):
        return False, f"output contains negative values (min={out.min():.4g})"
    return True, None


def _check_diagonal_zero(out):
    """Works for square matrices where X == Y was passed."""
    diag = np.diag(out)
    if not np.allclose(diag, 0.0, atol=1e-7):
        return False, f"diagonal is not zero when X==Y (max diag={diag.max():.4g})"
    return True, None


def _check_symmetric(out):
    if out.shape[0] != out.shape[1]:
        return False, "matrix is not square, symmetry check skipped"
    if not np.allclose(out, out.T, atol=1e-7):
        diff = np.abs(out - out.T).max()
        return False, f"matrix not symmetric when X==Y (max |out-out.T|={diff:.4g})"
    return True, None


def _combined_check(*fns):
    def chk(out):
        for f in fns:
            ok, msg = f(out)
            if not ok:
                return False, msg
        return True, None
    return chk


def _shape_check(N, M):
    def chk(out):
        if out.shape != (N, M):
            return False, f"wrong shape: expected ({N}, {M}), got {out.shape}"
        return True, None
    return chk


# ---- test cases -------------------------------------------------------------


# ── local runner ────────────────────────────────────────────────────────────
import numpy as np
from scipy.spatial.distance import cdist

def _run_tests_002():
    _tc = [
        # ---- tiny shapes --------------------------------------------------------
        _case("(1,3) vs (1,3) — scalar distance",
            inputs=lambda dev: {
                "X": np.array([[1.0, 2.0, 3.0]]),
                "Y": np.array([[4.0, 6.0, 3.0]]),
            },
            tol=(1e-10, 1e-10),
            check=_check_nonneg,
        ),
        _case("(3,1) vs (3,1) — column vectors",
            inputs=lambda dev: {
                "X": np.array([[1.0], [2.0], [3.0]]),
                "Y": np.array([[0.0], [1.0], [4.0]]),
            },
            tol=(1e-10, 1e-10),
            check=_check_nonneg,
        ),
        _case("(5,10) random",
            inputs=lambda dev: {
                "X": _rng(1).standard_normal((5, 10)),
                "Y": _rng(2).standard_normal((5, 10)),
            },
            check=_check_nonneg,
        ),

        # ---- medium shapes -------------------------------------------------------
        _case("(100,32) random",
            inputs=lambda dev: {
                "X": _rng(3).standard_normal((100, 32)),
                "Y": _rng(4).standard_normal((100, 32)),
            },
            check=_check_nonneg,
        ),

        # ---- identical X == Y: diagonal=0, symmetric ----------------------------
        _case("X==Y (4,4): diagonal zero",
            inputs=lambda dev: {
                "X": _rng(5).standard_normal((4, 4)),
                "Y": _rng(5).standard_normal((4, 4)),   # same seed → same array
            },
            check=_combined_check(_check_nonneg, _check_diagonal_zero, _check_symmetric),
        ),
        _case("X==Y (10,8): diagonal zero and symmetric",
            inputs=lambda dev: {
                "X": _rng(6).standard_normal((10, 8)),
                "Y": _rng(6).standard_normal((10, 8)),
            },
            check=_combined_check(_check_nonneg, _check_diagonal_zero, _check_symmetric),
        ),

        # ---- shape edge cases ---------------------------------------------------
        _case("N=1, M=10 (single query row)",
            inputs=lambda dev: {
                "X": _rng(7).standard_normal((1, 16)),
                "Y": _rng(8).standard_normal((10, 16)),
            },
            check=_combined_check(_shape_check(1, 10), _check_nonneg),
        ),
        _case("N=10, M=1 (single database row)",
            inputs=lambda dev: {
                "X": _rng(9).standard_normal((10, 16)),
                "Y": _rng(10).standard_normal((1, 16)),
            },
            check=_combined_check(_shape_check(10, 1), _check_nonneg),
        ),
        _case("D=1 (1-D points)",
            inputs=lambda dev: {
                "X": np.array([[0.0], [3.0], [5.0]]),
                "Y": np.array([[1.0], [4.0]]),
            },
            tol=(1e-10, 1e-10),
            check=_check_nonneg,
        ),

        # ---- catches missing sqrt ------------------------------------------------
        _case("known distances — catches missing sqrt",
            inputs=lambda dev: {
                "X": np.array([[0.0, 0.0], [3.0, 4.0]]),
                "Y": np.array([[0.0, 0.0]]),
            },
            tol=(1e-10, 1e-10),
        ),

        # ---- catches wrong axis for sum -----------------------------------------
        _case("catches wrong keepdims/axis — asymmetric N!=M",
            inputs=lambda dev: {
                "X": _rng(11).standard_normal((7, 5)),
                "Y": _rng(12).standard_normal((3, 5)),
            },
            check=_combined_check(_shape_check(7, 3), _check_nonneg),
        ),

        # ---- non-contiguous inputs (transpose) ----------------------------------
        _case("non-contiguous X (Fortran order)",
            inputs=lambda dev: {
                "X": np.asfortranarray(_rng(13).standard_normal((8, 6))),
                "Y": _rng(14).standard_normal((4, 6)),
            },
            check=_check_nonneg,
        ),
        _case("non-contiguous X (strided slice)",
            inputs=lambda dev: (
                lambda A: {"X": A[::2, :], "Y": A[1::2, :]}
            )(_rng(15).standard_normal((12, 8))),
            check=_check_nonneg,
        ),

        # ---- larger random case -------------------------------------------------
        _case("(50,64) vs (30,64)",
            inputs=lambda dev: {
                "X": _rng(16).standard_normal((50, 64)),
                "Y": _rng(17).standard_normal((30, 64)),
            },
            check=_combined_check(_shape_check(50, 30), _check_nonneg),
        ),

        # ---- zero matrix (all rows identical) -----------------------------------
        _case("all-zero X and Y (output must be all zero)",
            inputs=lambda dev: {
                "X": np.zeros((5, 4)),
                "Y": np.zeros((3, 4)),
            },
            tol=(0, 1e-12),
            check=_check_nonneg,
        ),
    ]
    sol = Solution002()
    passed = failed = 0
    for tc in _tc:
        name = tc["name"]
        inputs = tc["inputs"](None)
        try:
            out = sol.solve(**inputs)
            check = tc.get("check")
            if check:
                ok, msg = check(out)
                if not ok:
                    print(f"  FAIL  {name}: {msg}"); failed += 1; continue
            ref = cdist(inputs["X"], inputs["Y"])
            tol = tc.get("tolerance", (1e-9, 1e-9))
            if not np.allclose(out, ref, atol=tol[0], rtol=tol[1]):
                print(f"  FAIL  {name}: values don't match reference"); failed += 1; continue
            print(f"  PASS  {name}"); passed += 1
        except NotImplementedError:
            print(f"  SKIP  {name}: not implemented yet"); failed += 1
        except Exception as e:
            print(f"  ERROR {name}: {e}"); failed += 1
    print(f"\n{passed}/{passed+failed} tests passed")


In [134]:
_run_tests_002()

  PASS  (1,3) vs (1,3) — scalar distance
  PASS  (3,1) vs (3,1) — column vectors
  PASS  (5,10) random
  PASS  (100,32) random
  PASS  X==Y (4,4): diagonal zero
  PASS  X==Y (10,8): diagonal zero and symmetric
  PASS  N=1, M=10 (single query row)
  PASS  N=10, M=1 (single database row)
  PASS  D=1 (1-D points)
  PASS  known distances — catches missing sqrt
  PASS  catches wrong keepdims/axis — asymmetric N!=M
  PASS  non-contiguous X (Fortran order)
  PASS  non-contiguous X (strided slice)
  PASS  (50,64) vs (30,64)
  PASS  all-zero X and Y (output must be all zero)

15/15 tests passed


/tmp/ipykernel_1747/103308591.py:13: RuntimeWarning: invalid value encountered in sqrt
  out = np.sqrt( (X**2).sum(axis=1, keepdims=True)   - 2 * X @ Y.T  + (Y**2).sum(axis=1))


# 003 — One-Hot Encoding

**Difficulty:** Easy · **Topic:** NumPy fancy indexing

## Problem

Given an integer label array `labels` of shape `(N,)` with values in `[0, num_classes)` and an integer `num_classes`, produce a 2-D float32 array of shape `(N, num_classes)` where row `i` is all zeros except for a `1.0` at column `labels[i]`.

The canonical implementation uses NumPy fancy (advanced) indexing:

```python
out = np.zeros((N, num_classes), dtype=np.float32)
out[np.arange(N), labels] = 1.0
```

## Input

- `labels: np.ndarray` — shape `(N,)`, dtype `int64` (or any integer dtype), values in `[0, num_classes)`.
- `num_classes: int` — the number of classes `C`.

## Output

`np.ndarray` of shape `(N, num_classes)` and dtype `float32`.

## Constraints / Notes

- `num_classes` may be larger than `max(labels) + 1` (the extra columns remain zero).
- Each row must sum to exactly `1.0`.
- Exactly one column per row must be `1.0`; all others must be `0.0`.

## Examples

```python
>>> Solution().solve(np.array([0, 2, 1]), num_classes=3)
array([[1., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.]], dtype=float32)
```

## What you are learning

- NumPy advanced (fancy) indexing with `arange` + integer array
- In-place assignment with two index arrays
- Zero-initialisation with `np.zeros` and dtype specification

In [135]:
import numpy as np


class Solution003:
    """Produce a one-hot encoding matrix.

    labels: (N,) int — values in [0, num_classes)
    num_classes: int

    Returns: (N, num_classes) float32, each row has exactly one 1.0.
    """

    def solve(self, labels: np.ndarray, num_classes: int) -> np.ndarray:
        out = np.zeros((len(labels), num_classes), dtype=np.float32)

        out[np.arange(len(labels)), labels] = 1.0

        return out


Test your code here (run both blocks below):

In [136]:
# @title
import numpy as np


# No static_checks needed for this problem.


def _rng(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)


def _case(name, *, inputs, tol=(1e-7, 1e-7), check=None):
    return {"name": name, "inputs": inputs, "tolerance": tol, "check": check}


# ---- invariant checks -------------------------------------------------------

def _check_row_sum_one(out):
    row_sums = out.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-6):
        bad = np.where(~np.isclose(row_sums, 1.0, atol=1e-6))[0]
        return False, f"rows {bad[:5]} do not sum to 1 (got {row_sums[bad[:5]]})"
    return True, None


def _check_binary(out):
    unique = np.unique(out)
    if not set(unique).issubset({0.0, 1.0}):
        extra = [v for v in unique if v not in (0.0, 1.0)]
        return False, f"output contains non-0/1 values: {extra[:5]}"
    return True, None


def _check_shape(N, C):
    def chk(out):
        if out.shape != (N, C):
            return False, f"wrong shape: expected ({N}, {C}), got {out.shape}"
        return True, None
    return chk


def _check_dtype_float32(out):
    if out.dtype != np.float32:
        return False, f"expected float32, got {out.dtype}"
    return True, None


def _combined(*fns):
    def chk(out):
        for f in fns:
            ok, msg = f(out)
            if not ok:
                return False, msg
        return True, None
    return chk


# ---- test cases -------------------------------------------------------------


# ── local runner ────────────────────────────────────────────────────────────
import numpy as np

def _run_tests_003():
    _tc = [
        # ---- minimal cases ------------------------------------------------------
        _case("N=1, single label 0",
            inputs=lambda dev: {
                "labels": np.array([0], dtype=np.int64),
                "num_classes": 3,
            },
            check=_combined(_check_shape(1, 3), _check_row_sum_one, _check_binary, _check_dtype_float32),
        ),
        _case("N=1, single label == num_classes-1",
            inputs=lambda dev: {
                "labels": np.array([4], dtype=np.int64),
                "num_classes": 5,
            },
            check=_combined(_check_shape(1, 5), _check_row_sum_one, _check_binary, _check_dtype_float32),
        ),

        # ---- all same class -----------------------------------------------------
        _case("N=10, all label=2",
            inputs=lambda dev: {
                "labels": np.full(10, 2, dtype=np.int64),
                "num_classes": 5,
            },
            check=_combined(_check_shape(10, 5), _check_row_sum_one, _check_binary),
        ),

        # ---- all different labels (0..C-1) --------------------------------------
        _case("N=C, all different labels",
            inputs=lambda dev: {
                "labels": np.arange(6, dtype=np.int64),
                "num_classes": 6,
            },
            check=_combined(_check_shape(6, 6), _check_row_sum_one, _check_binary),
        ),

        # ---- num_classes larger than max label ----------------------------------
        _case("num_classes >> max_label (extra columns zero)",
            inputs=lambda dev: {
                "labels": np.array([0, 1, 2], dtype=np.int64),
                "num_classes": 10,
            },
            check=_combined(_check_shape(3, 10), _check_row_sum_one, _check_binary),
        ),

        # ---- large N ------------------------------------------------------------
        _case("N=1000, random labels",
            inputs=lambda dev: {
                "labels": _rng(1).integers(0, 20, size=1000).astype(np.int64),
                "num_classes": 20,
            },
            check=_combined(_check_shape(1000, 20), _check_row_sum_one, _check_binary),
        ),

        # ---- catches off-by-one in column index ---------------------------------
        _case("labels=[0,1,2,3,4] with num_classes=5",
            inputs=lambda dev: {
                "labels": np.array([0, 1, 2, 3, 4], dtype=np.int64),
                "num_classes": 5,
            },
            check=_combined(_check_shape(5, 5), _check_row_sum_one, _check_binary),
        ),

        # ---- catches wrong dtype ------------------------------------------------
        _case("output dtype must be float32",
            inputs=lambda dev: {
                "labels": np.array([0, 1], dtype=np.int64),
                "num_classes": 3,
            },
            check=_check_dtype_float32,
        ),

        # ---- larger num_classes -------------------------------------------------
        _case("N=8, num_classes=100 (sparse)",
            inputs=lambda dev: {
                "labels": _rng(2).integers(0, 100, size=8).astype(np.int64),
                "num_classes": 100,
            },
            check=_combined(_check_shape(8, 100), _check_row_sum_one, _check_binary),
        ),

        # ---- non-int64 label dtype (int32) --------------------------------------
        _case("labels dtype int32",
            inputs=lambda dev: {
                "labels": np.array([1, 0, 2], dtype=np.int32),
                "num_classes": 4,
            },
            check=_combined(_check_shape(3, 4), _check_row_sum_one, _check_binary, _check_dtype_float32),
        ),

        # ---- specific values check (ground truth) --------------------------------
        _case("known output (3-class, 3 samples)",
            inputs=lambda dev: {
                "labels": np.array([0, 2, 1], dtype=np.int64),
                "num_classes": 3,
            },
            tol=(0, 0),
        ),

        # ---- catches overwrite bug (row repeated label) -------------------------
        _case("repeated labels should not corrupt other rows",
            inputs=lambda dev: {
                "labels": np.array([1, 1, 1, 0, 2], dtype=np.int64),
                "num_classes": 3,
            },
            check=_combined(_check_row_sum_one, _check_binary),
        ),
    ]
    sol = Solution003()
    passed = failed = 0
    for tc in _tc:
        name = tc["name"]
        inputs = tc["inputs"](None)
        try:
            out = sol.solve(**inputs)
            check = tc.get("check")
            if check:
                ok, msg = check(out)
                if not ok:
                    print(f"  FAIL  {name}: {msg}"); failed += 1; continue
            N = inputs["labels"].shape[0]
            C = inputs["num_classes"]
            ref = np.zeros((N, C), dtype=np.float32)
            ref[np.arange(N), inputs["labels"]] = 1.0
            tol = tc.get("tolerance", (1e-7, 1e-7))
            if not np.allclose(out, ref, atol=tol[0], rtol=tol[1]):
                print(f"  FAIL  {name}: values don't match reference"); failed += 1; continue
            print(f"  PASS  {name}"); passed += 1
        except NotImplementedError:
            print(f"  SKIP  {name}: not implemented yet"); failed += 1
        except Exception as e:
            print(f"  ERROR {name}: {e}"); failed += 1
    print(f"\n{passed}/{passed+failed} tests passed")


In [137]:
_run_tests_003()

  PASS  N=1, single label 0
  PASS  N=1, single label == num_classes-1
  PASS  N=10, all label=2
  PASS  N=C, all different labels
  PASS  num_classes >> max_label (extra columns zero)
  PASS  N=1000, random labels
  PASS  labels=[0,1,2,3,4] with num_classes=5
  PASS  output dtype must be float32
  PASS  N=8, num_classes=100 (sparse)
  PASS  labels dtype int32
  PASS  known output (3-class, 3 samples)
  PASS  repeated labels should not corrupt other rows

12/12 tests passed


# 004 — Cross-Entropy Loss

**Difficulty:** Easy · **Topic:** Torch losses

## Problem

Implement cross-entropy loss from scratch given a batch of logits and integer class targets.

Given `logits` of shape `(B, C)` and `targets` of shape `(B,)` with integer values in `[0, C)`,
compute the per-sample cross-entropy:

```
CE_b = -log_softmax(logits[b])[targets[b]]
     = -(logits[b, targets[b]] - log(sum_c exp(logits[b])))
```

where the log-softmax is computed in a numerically stable way (subtract the row max before exponentiating).

The `reduction` argument controls the output:
- `"mean"` — scalar: `mean(CE_b)`
- `"sum"`  — scalar: `sum(CE_b)`
- `"none"` — tensor of shape `(B,)`

You **must not** call `F.cross_entropy`, `F.nll_loss`, or `nn.CrossEntropyLoss`.

## Input

- `logits: torch.Tensor` — shape `(B, C)`, floating dtype.
- `targets: torch.Tensor` — shape `(B,)`, dtype `torch.long`, values in `[0, C)`.
- `reduction: str` — one of `"mean"`, `"sum"`, `"none"` (default `"mean"`).

## Output

- If `reduction` is `"mean"` or `"sum"`: a scalar tensor of the same float dtype as `logits`.
- If `reduction` is `"none"`: a tensor of shape `(B,)` of the same float dtype as `logits`.

## Examples

```python
>>> logits = torch.tensor([[2.0, 1.0, 0.1]])
>>> targets = torch.tensor([0])
>>> Solution().solve(logits, targets, reduction="mean")
tensor(0.4170)
```

In [138]:
import torch
import numpy as np

class Solution004:
    """Implement cross-entropy loss.
    Do NOT use F.cross_entropy, F.nll_loss, or nn.CrossEntropyLoss.
    """
    def solve(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
        reduction: str = "mean",
    ) -> torch.Tensor:
        dtype = logits.dtype

        one_hot = torch.zeros(logits.shape[0],logits.shape[1], dtype=dtype)

        one_hot[torch.arange(logits.shape[0]),   targets] = 1.0

        x_max = logits.max(dim=1, keepdim=True).values

        log_softmax = (logits - x_max) - torch.log(

            torch.exp(logits - x_max).sum(dim=1, keepdim=True)

        )
        before_reduction = -(one_hot * log_softmax).sum(dim=1)

        if reduction == "mean":

            return before_reduction.mean()

        elif reduction == "sum":

            return before_reduction.sum()

        else:

            return before_reduction

Test your code here (run both blocks below):

In [139]:
# @title
import torch
import torch.nn.functional as F

try:
    from server.ast_checks import find_forbidden
except ModuleNotFoundError:
    find_forbidden = lambda src, banned: []  # local fallback


_BANNED_CALLS = [
    "F.cross_entropy",
    "F.nll_loss",
    "torch.nn.functional.cross_entropy",
    "torch.nn.functional.nll_loss",
    "nn.CrossEntropyLoss",
    "CrossEntropyLoss",
]


def static_checks(user_source: str):
    issues = []
    hits = find_forbidden(user_source, _BANNED_CALLS)
    if hits:
        issues.append((False, f"Calls to {', '.join(hits)} are not allowed — implement cross-entropy manually."))
    return issues


def _gen(seed: int):
    return torch.Generator().manual_seed(seed)


def _rand_logits(B, C, seed, dtype=torch.float32, device="cpu", scale=1.0):
    g = _gen(seed)
    t = torch.randn(B, C, generator=g, dtype=torch.float32) * scale
    return t.to(device=device, dtype=dtype)


def _rand_targets(B, C, seed, device="cpu"):
    g = _gen(seed)
    return torch.randint(0, C, (B,), generator=g).to(device=device)


def _case(name, *, inputs, tol=(1e-3, 1e-4), check=None):
    return {"name": name, "inputs": inputs, "tolerance": tol, "check": check}


# ---- reference cross-entropy comparison ------------------------------------

def _ref_ce(logits, targets, reduction):
    """Compute CE using F.cross_entropy as ground-truth oracle."""
    return F.cross_entropy(logits.float(), targets, reduction=reduction).to(logits.dtype)


def _make_ref_check(reduction):
    def chk(out):
        # We cannot access inputs inside check, so just validate the shape/finiteness
        if not torch.isfinite(out).all():
            return False, "output contains non-finite values"
        return True, None
    return chk


# ---- test cases -------------------------------------------------------------


# ── local runner ────────────────────────────────────────────────────────────
import torch.nn.functional as F

def _run_tests_004():
    _tc = [
        # Basic mean reduction
        _case("basic mean (B=4, C=10)",
            inputs=lambda dev: {
                "logits": _rand_logits(4, 10, 1, device=dev),
                "targets": _rand_targets(4, 10, 1, device=dev),
                "reduction": "mean",
            },
        ),

        # sum reduction
        _case("sum reduction (B=4, C=10)",
            inputs=lambda dev: {
                "logits": _rand_logits(4, 10, 2, device=dev),
                "targets": _rand_targets(4, 10, 2, device=dev),
                "reduction": "sum",
            },
        ),

        # none reduction — output shape (B,)
        _case("none reduction shape (B=4, C=10)",
            inputs=lambda dev: {
                "logits": _rand_logits(4, 10, 3, device=dev),
                "targets": _rand_targets(4, 10, 3, device=dev),
                "reduction": "none",
            },
            check=lambda out: (tuple(out.shape) == (4,), f"expected shape (4,), got {tuple(out.shape)}"),
        ),

        # B=1 edge case
        _case("B=1",
            inputs=lambda dev: {
                "logits": _rand_logits(1, 5, 4, device=dev),
                "targets": _rand_targets(1, 5, 4, device=dev),
                "reduction": "mean",
            },
        ),

        # C=2 binary classification
        _case("C=2 (binary)",
            inputs=lambda dev: {
                "logits": _rand_logits(8, 2, 5, device=dev),
                "targets": _rand_targets(8, 2, 5, device=dev),
                "reduction": "mean",
            },
        ),

        # C=1000 (ImageNet-like)
        _case("C=1000 (B=16)",
            inputs=lambda dev: {
                "logits": _rand_logits(16, 1000, 6, device=dev),
                "targets": _rand_targets(16, 1000, 6, device=dev),
                "reduction": "mean",
            },
        ),

        # Perfect predictions: very large logit at target class → loss ≈ 0
        _case("perfect predictions (loss ≈ 0)",
            inputs=lambda dev: {
                "logits": (lambda t: t.scatter_(1, torch.zeros(t.size(0), 1, dtype=torch.long, device=dev), 100.0))(
                    torch.zeros(8, 10, device=dev)
                ),
                "targets": torch.zeros(8, dtype=torch.long, device=dev),
                "reduction": "mean",
            },
            check=lambda out: (out.item() < 1e-3, f"expected loss≈0 for perfect preds, got {out.item():.4g}"),
        ),

        # Max-wrong predictions: large logit on all wrong classes → high loss
        _case("max-wrong predictions (high loss)",
            inputs=lambda dev: {
                "logits": (lambda t: t.scatter_(1, torch.ones(t.size(0), 1, dtype=torch.long, device=dev), 100.0))(
                    torch.zeros(8, 10, device=dev)
                ),
                "targets": torch.zeros(8, dtype=torch.long, device=dev),  # target is 0, but logit 1 is boosted
                "reduction": "mean",
            },
            check=lambda out: (out.item() > 90.0, f"expected high loss for max-wrong preds, got {out.item():.4g}"),
        ),

        # Large magnitudes — numerical stability
        _case("large logit magnitudes (scale=1e3)",
            inputs=lambda dev: {
                "logits": _rand_logits(8, 20, 7, device=dev, scale=1e3),
                "targets": _rand_targets(8, 20, 7, device=dev),
                "reduction": "mean",
            },
        ),

        # float64 precision
        _case("float64 precision",
            inputs=lambda dev: {
                "logits": _rand_logits(6, 15, 8, dtype=torch.float64, device=dev),
                "targets": _rand_targets(6, 15, 8, device=dev),
                "reduction": "mean",
            },
            tol=(1e-9, 1e-12),
        ),

        # float64 + none reduction
        _case("float64 none reduction",
            inputs=lambda dev: {
                "logits": _rand_logits(5, 12, 9, dtype=torch.float64, device=dev),
                "targets": _rand_targets(5, 12, 9, device=dev),
                "reduction": "none",
            },
            tol=(1e-9, 1e-12),
        ),

        # B=64 mean reduction
        _case("B=64 mean",
            inputs=lambda dev: {
                "logits": _rand_logits(64, 10, 10, device=dev),
                "targets": _rand_targets(64, 10, 10, device=dev),
                "reduction": "mean",
            },
        ),

        # B=64 sum reduction
        _case("B=64 sum",
            inputs=lambda dev: {
                "logits": _rand_logits(64, 10, 11, device=dev),
                "targets": _rand_targets(64, 10, 11, device=dev),
                "reduction": "sum",
            },
        ),
    ]
    sol = Solution004()
    passed = failed = 0
    for tc in _tc:
        name = tc["name"]
        inputs = tc["inputs"]("cpu")
        try:
            out = sol.solve(**inputs)
            check = tc.get("check")
            if check:
                ok, msg = check(out)
                if not ok:
                    print(f"  FAIL  {name}: {msg}"); failed += 1; continue
            ref = F.cross_entropy(inputs["logits"].float(), inputs["targets"],
                                  reduction=inputs.get("reduction", "mean")).to(inputs["logits"].dtype)
            tol = tc.get("tolerance", (1e-3, 1e-4))
            if not torch.allclose(out.float(), ref.float(), atol=tol[0], rtol=tol[1]):
                print(f"  FAIL  {name}: got {out.item():.6f}, expected {ref.item():.6f}"); failed += 1; continue
            print(f"  PASS  {name}"); passed += 1
        except NotImplementedError:
            print(f"  SKIP  {name}: not implemented yet"); failed += 1
        except Exception as e:
            print(f"  ERROR {name}: {e}"); failed += 1
    print(f"\n{passed}/{passed+failed} tests passed")


In [140]:
_run_tests_004()

  PASS  basic mean (B=4, C=10)
  PASS  sum reduction (B=4, C=10)
  PASS  none reduction shape (B=4, C=10)
  PASS  B=1
  PASS  C=2 (binary)
  PASS  C=1000 (B=16)
  PASS  perfect predictions (loss ≈ 0)
  PASS  max-wrong predictions (high loss)
  PASS  large logit magnitudes (scale=1e3)
  FAIL  float64 precision: got 3.488638, expected 3.488638
  PASS  float64 none reduction
  PASS  B=64 mean
  PASS  B=64 sum

12/13 tests passed


# 005 — Gradient Clipping

**Difficulty:** Easy · **Topic:** Training utilities

## Problem

Implement **gradient norm clipping** as used in `torch.nn.utils.clip_grad_norm_`.

Given a list of parameter tensors (each with a `.grad` attribute), an optional `norm_type`, and a `max_norm`:

1. Collect all non-`None` gradients.
2. Compute the **total gradient norm**:
   - For `norm_type != inf`: `total_norm = (Σ ||g||^norm_type)^(1/norm_type)`
   - For `norm_type == inf`: `total_norm = max(||g||_inf)` across all grads
3. If `total_norm > max_norm`: scale every gradient **in-place** by `max_norm / total_norm`.
4. Return `total_norm` (as a Python `float`, **before clipping**).

## Input

- `parameters: list` — list of tensors, each with a `.grad` attribute (may be `None`)
- `max_norm: float` — maximum allowed norm
- `norm_type: float` — order of the norm (default `2.0`); use `float('inf')` for infinity norm

## Output

`float` — the total gradient norm **before** clipping.

## Constraints / Notes

- Do **not** use `nn.utils.clip_grad_norm_` or `nn.utils.clip_grad_value_`.
- Gradients with `.grad is None` are skipped.
- If `parameters` is empty or all grads are `None`, return `0.0`.
- Gradients must be modified **in-place**.

## Examples

```python
>>> p = torch.tensor([1.0, 0.0, 0.0], requires_grad=True)
>>> p.grad = torch.tensor([3.0, 4.0, 0.0])
>>> norm = Solution().solve([p], max_norm=1.0)
>>> norm
5.0
>>> p.grad  # clipped to norm 1
tensor([0.6000, 0.8000, 0.0000])
```

## What you are learning

- Computing global gradient norms across parameters
- In-place tensor modification
- L1, L2, and L-infinity norms

In [141]:
import torch

class Solution005:
    def solve(self, parameters, max_norm, norm_type=2.0):
        grads = []
        # iterating over parameters
        for p in parameters:

            if p.grad is not None:

                grads.append(p.grad)

        if len(grads) == 0:

            return 0.0

        if norm_type == float('inf'):

            total_norm = 0.0

            for g in grads:

                current = g.abs().max().item()

                if current > total_norm:

                    total_norm = current

        else:

            total_norm = 0.0

            for g in grads:

                #total norm

                total_norm = total_norm + g.norm(norm_type).item() ** norm_type

            total_norm = total_norm ** (1.0 / norm_type)

        # checking the condition
        if total_norm > max_norm:

            clip_coef = max_norm / total_norm

            for g in grads:

                g.mul_(clip_coef)

        return total_norm

Test your code here (run both blocks below):

In [142]:
# @title
import math
import torch

try:
    from server.ast_checks import find_forbidden
except ModuleNotFoundError:
    find_forbidden = lambda src, banned: []  # local fallback


_BANNED_CALLS = [
    "clip_grad_norm_",
    "clip_grad_value_",
    "nn.utils.clip_grad_norm_",
    "nn.utils.clip_grad_value_",
]


def static_checks(user_source: str):
    issues = []
    hits = find_forbidden(user_source, _BANNED_CALLS)
    if hits:
        issues.append(
            (False, f"Calls to {', '.join(hits)} are not allowed — implement gradient clipping manually.")
        )
    return issues


# ---------------------------------------------------------------------------
# Helpers

class _Param:
    """Minimal parameter stub with a .grad attribute."""
    def __init__(self, grad):
        self.grad = grad


def _gen(seed: int) -> torch.Generator:
    return torch.Generator().manual_seed(seed)


def _randn(shape, seed, dtype=torch.float32, device="cpu"):
    g = _gen(seed)
    return torch.randn(*shape, generator=g, dtype=torch.float32).to(device=device, dtype=dtype)


def _case(name, *, inputs, tol=(1e-4, 1e-5), check=None):
    return {"name": name, "inputs": inputs, "tolerance": tol, "check": check}


# ---------------------------------------------------------------------------
# Note on test design
#
# The runner calls inputs(dev) ONCE, then shallow-copies for ref and user.
# _Param objects are NOT torch.Tensors, so they are shared between ref_inputs
# and user_inputs.  To keep tests deterministic, every test that involves
# clipping uses max_norm >= total_norm so NO in-place mutation occurs and
# both ref and user see identical gradient tensors and return the same norm.
#
# Tests that specifically verify clipping (total_norm > max_norm) use
# pre-computed norms so max_norm is set to exactly total_norm * 2, ensuring
# no clipping, while the "does clip" logic is validated by other means via
# the check function examining the return value.
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Test cases


# ── local runner ────────────────────────────────────────────────────────────
import math, copy, torch

def _run_tests_005():
    _tc = [
        # --- all grads small (no clipping), verify correct norm returned ---
        _case("norm-2 small grads (no clip)",
            inputs=lambda dev: {
                "parameters": [
                    _Param(_randn((4,), 1, device=dev)),
                    _Param(_randn((3, 3), 2, device=dev)),
                ],
                "max_norm": 1e10,
                "norm_type": 2.0,
            },
        ),

        # --- single parameter ---
        _case("single param norm-2",
            inputs=lambda dev: {
                "parameters": [_Param(torch.tensor([3.0, 4.0], device=dev))],
                "max_norm": 1e10,
                "norm_type": 2.0,
            },
            # ||[3,4]|| = 5
        ),

        # --- empty list → return 0.0 ---
        _case("empty parameters list",
            inputs=lambda dev: {
                "parameters": [],
                "max_norm": 1.0,
                "norm_type": 2.0,
            },
            check=lambda out: (
                out == 0.0,
                None if out == 0.0 else f"expected 0.0 for empty list, got {out}"
            ),
        ),

        # --- all grads None → return 0.0 ---
        _case("all grads None",
            inputs=lambda dev: {
                "parameters": [_Param(None), _Param(None)],
                "max_norm": 1.0,
                "norm_type": 2.0,
            },
            check=lambda out: (
                out == 0.0,
                None if out == 0.0 else f"expected 0.0 when all grads are None, got {out}"
            ),
        ),

        # --- mixed None and non-None grads ---
        _case("mixed None and non-None grads",
            inputs=lambda dev: {
                "parameters": [
                    _Param(None),
                    _Param(torch.tensor([3.0, 4.0], device=dev)),
                    _Param(None),
                ],
                "max_norm": 1e10,
                "norm_type": 2.0,
            },
            # Only [3,4] contributes → norm = 5
        ),

        # --- norm_type=1 ---
        _case("norm_type=1 (L1 norm)",
            inputs=lambda dev: {
                "parameters": [
                    _Param(torch.tensor([1.0, 2.0, 3.0], device=dev)),
                    _Param(torch.tensor([4.0], device=dev)),
                ],
                "max_norm": 1e10,
                "norm_type": 1.0,
            },
            # sum = 1+2+3+4 = 10
        ),

        # --- norm_type=inf ---
        _case("norm_type=inf (Linf norm)",
            inputs=lambda dev: {
                "parameters": [
                    _Param(torch.tensor([1.0, 5.0, 2.0], device=dev)),
                    _Param(torch.tensor([3.0, 4.0], device=dev)),
                ],
                "max_norm": 1e10,
                "norm_type": float("inf"),
            },
            # max absolute = 5
        ),

        # --- fp32 multi-param ---
        _case("fp32 multiple params norm-2",
            inputs=lambda dev: {
                "parameters": [
                    _Param(_randn((8, 8), 10, device=dev)),
                    _Param(_randn((8,), 11, device=dev)),
                    _Param(_randn((4, 4), 12, device=dev)),
                ],
                "max_norm": 1e10,
                "norm_type": 2.0,
            },
        ),

        # --- fp64 ---
        _case("fp64 grads norm-2",
            inputs=lambda dev: {
                "parameters": [
                    _Param(_randn((4, 4), 20, dtype=torch.float64, device=dev)),
                    _Param(_randn((4,), 21, dtype=torch.float64, device=dev)),
                ],
                "max_norm": 1e10,
                "norm_type": 2.0,
            },
            tol=(1e-7, 1e-9),
        ),

        # --- 2D grad tensors ---
        _case("2D grad tensors",
            inputs=lambda dev: {
                "parameters": [
                    _Param(torch.tensor([[1.0, 0.0], [0.0, 1.0]], device=dev)),
                    _Param(torch.tensor([[2.0, 0.0], [0.0, 2.0]], device=dev)),
                ],
                "max_norm": 1e10,
                "norm_type": 2.0,
            },
            # norms: sqrt(2), 2*sqrt(2); total = sqrt(2+8) = sqrt(10)
        ),

        # --- verify large norm return is correct (no-clip version) ---
        _case("large grads norm-2 (no-clip sanity)",
            inputs=lambda dev: {
                "parameters": [
                    _Param(torch.tensor([100.0, 0.0], device=dev)),
                    _Param(torch.tensor([0.0, 100.0], device=dev)),
                ],
                "max_norm": 1e10,
                "norm_type": 2.0,
            },
            # norm = sqrt(100^2 + 100^2) = 100*sqrt(2) ≈ 141.42
        ),
    ]
    sol = Solution005()
    passed = failed = 0
    for tc in _tc:
        name = tc["name"]
        inputs_raw = tc["inputs"]("cpu")
        # deep-copy params so ref and user don't share grad tensors
        params_copy = [_Param(copy.deepcopy(p.grad)) for p in inputs_raw["parameters"]]
        inputs_user = {**inputs_raw, "parameters": params_copy}
        try:
            out = sol.solve(**inputs_user)
            check = tc.get("check")
            if check:
                ok, msg = check(out)
                if not ok:
                    print(f"  FAIL  {name}: {msg}"); failed += 1; continue
            # reference norm
            grads = [p.grad for p in inputs_raw["parameters"] if p.grad is not None]
            nt = inputs_raw.get("norm_type", 2.0)
            if not grads:
                ref_norm = 0.0
            elif nt == float("inf"):
                ref_norm = max(g.abs().max().item() for g in grads)
            else:
                ref_norm = sum(g.norm(nt).item()**nt for g in grads)**(1.0/nt)
            tol = tc.get("tolerance", (1e-4, 1e-5))
            if not math.isclose(out, ref_norm, rel_tol=tol[1], abs_tol=tol[0]):
                print(f"  FAIL  {name}: got {out:.6f}, expected {ref_norm:.6f}"); failed += 1; continue
            print(f"  PASS  {name}"); passed += 1
        except NotImplementedError:
            print(f"  SKIP  {name}: not implemented yet"); failed += 1
        except Exception as e:
            print(f"  ERROR {name}: {e}"); failed += 1
    print(f"\n{passed}/{passed+failed} tests passed")



In [143]:
_run_tests_005()

  PASS  norm-2 small grads (no clip)
  PASS  single param norm-2
  PASS  empty parameters list
  PASS  all grads None
  PASS  mixed None and non-None grads
  PASS  norm_type=1 (L1 norm)
  PASS  norm_type=inf (Linf norm)
  PASS  fp32 multiple params norm-2
  PASS  fp64 grads norm-2
  PASS  2D grad tensors
  PASS  large grads norm-2 (no-clip sanity)

11/11 tests passed
